In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score

# =========================
# PATHS (Train, TEST DATA)
# =========================
TRAIN_PATH = ""
TEST_PATH  = ""

# =========================
# 1) LOAD TRAIN
# =========================
train = pd.read_csv(TRAIN_PATH, index_col=0)

# force string col names just in case
train.columns = train.columns.astype(str)

# required columns
required_train = ["1002", "1003", "1004", "label"]
missing_train = [c for c in required_train if c not in train.columns]
if missing_train:
    raise ValueError(f"Missing columns in TRAIN: {missing_train}")

train["chr"]   = train["1004"].astype(int)
train["start"] = train["1002"].astype(int)
train["end"]   = train["1003"].astype(int)

# locus id
train["locus"] = (
    train["chr"].astype(str) + "_" +
    train["start"].astype(str) + "_" +
    train["end"].astype(str)
)

# CNV label in training (DEL=1, DUP=2)
train["is_del"] = (train["label"] == 1).astype(int)
train["is_dup"] = (train["label"] == 2).astype(int)
train["is_cnv"] = train["label"].isin([1, 2]).astype(int)

# =========================
# 2) PER-LOCUS TRAIN FREQUENCY
# =========================
# total occurrences of that locus across samples/windows
locus_total = train.groupby("locus").size().rename("n_total")

# how many times this locus was DEL / DUP / CNV in train
locus_del = train.groupby("locus")["is_del"].sum().rename("n_del")
locus_dup = train.groupby("locus")["is_dup"].sum().rename("n_dup")
locus_cnv = train.groupby("locus")["is_cnv"].sum().rename("n_cnv")

locus_stats = pd.concat([locus_total, locus_del, locus_dup, locus_cnv], axis=1).fillna(0)

# frequency of being labeled CNV at that locus in training
locus_stats["freq_cnv"] = locus_stats["n_cnv"] / locus_stats["n_total"]

# also class-specific frequencies (optional, sometimes useful)
locus_stats["freq_del"] = locus_stats["n_del"] / locus_stats["n_total"]
locus_stats["freq_dup"] = locus_stats["n_dup"] / locus_stats["n_total"]

# =========================
# 3) LOAD TEST (true vs predicted)
# =========================
test = pd.read_csv(TEST_PATH, index_col=0)
test.columns = test.columns.astype(str)

required_test = ["1002", "1003", "1004", "True Labels", "Predicted Labels"]
missing_test = [c for c in required_test if c not in test.columns]
if missing_test:
    raise ValueError(f"Missing columns in TEST: {missing_test}")

test["chr"]   = test["1004"].astype(int)
test["start"] = test["1002"].astype(int)
test["end"]   = test["1003"].astype(int)

test["locus"] = (
    test["chr"].astype(str) + "_" +
    test["start"].astype(str) + "_" +
    test["end"].astype(str)
)

# map training freq into test
test = test.merge(locus_stats[["freq_cnv", "freq_del", "freq_dup"]], how="left", left_on="locus", right_index=True)

# loci never seen in training => freq = 0
test[["freq_cnv", "freq_del", "freq_dup"]] = test[["freq_cnv", "freq_del", "freq_dup"]].fillna(0.0)

# =========================
# 4) STRATA DEFINITIONS
# =========================
def stratum_from_freq(freq):
    if freq == 0:
        return "Never (0)"
    elif freq <= 0.05:
        return "Rare (0–5%)"
    elif freq <= 0.50:
        return "Often (5–50%)"
    else:
        return "Majority (>50%)"

test["stratum_cnv"] = test["freq_cnv"].apply(stratum_from_freq)

# =========================
# 5) EVALUATION HELPERS
# =========================
def eval_binary(y_true, y_pred):
    """Return precision, recall, f1 for a binary task."""
    return {
        "n": len(y_true),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

# binary CNV vs no-call
test["true_cnv"] = test["True Labels"].isin([1,2]).astype(int)
test["pred_cnv"] = test["Predicted Labels"].isin([1,2]).astype(int)

# DEL one-vs-rest (DEL vs not DEL)
test["true_del"] = (test["True Labels"] == 1).astype(int)
test["pred_del"] = (test["Predicted Labels"] == 1).astype(int)

# DUP one-vs-rest (DUP vs not DUP)
test["true_dup"] = (test["True Labels"] == 2).astype(int)
test["pred_dup"] = (test["Predicted Labels"] == 2).astype(int)

# =========================
# 6) REPORT PER STRATUM (CNV windows only)
# =========================
# Reviewer asked: stratify *test deletion/duplication windows*
cnv_only = test[test["True Labels"].isin([1,2])].copy()

results = []
for s, df_s in cnv_only.groupby("stratum_cnv"):
    # Evaluate "CNV detected?" within true-CNV windows => this is basically recall for CNV calls,
    # but we still report precision/f1 for completeness using pred_cnv (can be all 1s sometimes).
    r_cnv = eval_binary(df_s["true_cnv"], df_s["pred_cnv"])

    # Evaluate DEL recall/F1 on true DEL windows inside this stratum
    df_del = df_s[df_s["True Labels"] == 1]
    r_del = eval_binary(df_del["true_del"], df_del["pred_del"]) if len(df_del) else {"n":0,"precision":np.nan,"recall":np.nan,"f1":np.nan}

    # Evaluate DUP recall/F1 on true DUP windows inside this stratum
    df_dup = df_s[df_s["True Labels"] == 2]
    r_dup = eval_binary(df_dup["true_dup"], df_dup["pred_dup"]) if len(df_dup) else {"n":0,"precision":np.nan,"recall":np.nan,"f1":np.nan}

    results.append({
        "Stratum": s,
        "n_true_CNV": len(df_s),

        "CNV_recall": r_cnv["recall"],
        "CNV_f1": r_cnv["f1"],

        "n_true_DEL": r_del["n"],
        "DEL_recall": r_del["recall"],
        "DEL_f1": r_del["f1"],

        "n_true_DUP": r_dup["n"],
        "DUP_recall": r_dup["recall"],
        "DUP_f1": r_dup["f1"],
    })

results_df = pd.DataFrame(results).sort_values(
    by="Stratum",
    key=lambda x: x.map({"Never (0)":0, "Rare (0–5%)":1, "Often (5–50%)":2, "Majority (>50%)":3})
)

print("\n=== Stratified performance on TRUE CNV windows (DEL+DUP) ===")
print(results_df.to_string(index=False))

# Optional: show how many CNV test windows fall in each stratum
print("\n=== Stratum counts (TRUE CNV windows only) ===")
print(cnv_only["stratum_cnv"].value_counts())
